In [ ]:
import gym
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
from gym import spaces
from scipy.spatial import distance
import random
import datetime
from datetime import timezone
from datetime import datetime

In [ ]:
import numpy as np
from datetime import datetime, timezone
def satellite_coverage_check(r1, r2, Re=6371, threshold=500):
        r1, r2 = np.array(r1), np.array(r2)
        d = np.linalg.norm(r1 - r2)
        d_min = np.linalg.norm(np.cross(r1, r2)) / d
        return d_min > Re - threshold  # relxing the constraint

def is_on_earth(position, epsilon=10):
    R_E = 6371  #
    distance_from_center = np.linalg.norm(position)
    return abs(distance_from_center - R_E) <= epsilon
def julian_date(utc_time):
    """Convert UTC time to Julian Date."""
    unix_epoch = datetime(1970, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
    delta_seconds = (utc_time - unix_epoch).total_seconds()
    julian_date = 2440587.5 + delta_seconds / 86400.0
    return julian_date

def gst_from_julian(julian_date):
    """Compute Greenwich Sidereal Time from Julian Date."""
    T = (julian_date - 2451545.0) / 36525.0
    # Compute GST in degrees, modulo 360 to stay within bounds
    GST = 280.46061837 + 360.98564736629 * (julian_date - 2451545.0) + 0.000387933 * T**2 - (T**3 / 38710000.0)
    GST = GST % 360.0  # Ensure GST is within [0, 360] degrees
    return np.radians(GST)



def geodetic_to_eci(lat_deg, lon_deg, altitude_km=0, utc_time=None):

      if utc_time is None:
        utc_time = datetime.utcnow().replace(tzinfo=timezone.utc)

      lat = np.radians(lat_deg)
      lon = np.radians(lon_deg)

      R_E = 6378.137

    # Compute the Julian Date from the given UTC time
      jd = julian_date(utc_time)

      gst = gst_from_julian(jd)

      lon_eci = lon + gst  # Longitude in the inertial frame (ECI)

      x = (R_E + altitude_km) * np.cos(lat) * np.cos(lon_eci)
      y = (R_E + altitude_km) * np.cos(lat) * np.sin(lon_eci)
      z = (R_E + altitude_km) * np.sin(lat)

      return x, y, z



def is_on_earth(position, epsilon=1e-3):#to check wether the current node is a sallite or a ground point while routing
    R_E = 6371
    distance_from_center = np.linalg.norm(position)
    return abs(distance_from_center - R_E) >= epsilon


In [ ]:

import heapq
import numpy as np
from math import radians, sin, cos, sqrt, atan2

def great_circle_distance(pos1, pos2, R_E=6371):

     #the greatcircle dist
    lat1 = np.arcsin(pos1[2] / np.linalg.norm(pos1))
    lon1 = np.arctan2(pos1[1], pos1[0])
    lat2 = np.arcsin(pos2[2] / np.linalg.norm(pos2))
    lon2 = np.arctan2(pos2[1], pos2[0])

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R_E * c
def a_star_routing(source, target, satellite_positions, R_E=6371.8):

    # reverse lookup
    position_to_index = {tuple(v): k for k, v in satellite_positions.items()}

    if tuple(source) not in position_to_index or tuple(target) not in position_to_index:
        raise ValueError("src or tgt position not in satellite_positions.")

    open_set = []
    heapq.heappush(open_set, (0, source))  # (f_cost, current_node)

    g_cost = {tuple(pos): float('inf') for pos in satellite_positions.values()}
    g_cost[tuple(source)] = 0

    came_from = {}
    visited_sats = set()

    def edge_cost(current, neighbor):

        return great_circle_distance(np.array(current), np.array(neighbor))

    def heuristic_cost_estimate(current, goal, method="delay"):

        if method == "delay":
            return great_circle_distance(np.array(current), np.array(goal))

    def get_neighbors(current, satellite_positions, max_range=2000):

        neighbors = []
        current_pos = np.array(current)
        for pos in satellite_positions.values():
            if satellite_coverage_check(current, pos):
                neighbors.append(tuple(pos))
        return neighbors
    def reconstruct_path(came_from, current):

        path = [current]
        while current in came_from:
            current = came_from[current]
            path.insert(0, current)

        hop_distances = [edge_cost(path[i], path[i + 1]) for i in range(len(path) - 1)]
        total_distance = sum(hop_distances)

        print("hop dists:", hop_distances)
        return [position_to_index[tuple(p)] for p in path], total_distance

    while open_set:
        _, current = heapq.heappop(open_set)
        visited_sats.add(position_to_index[tuple(current)])

        if np.array_equal(current, target):
            path_indices, distance = reconstruct_path(came_from, tuple(current))
            if distance > R_E:
                print(" total  dist is  > re : the path involves multiple hops.")
            return distance, list(visited_sats), path_indices

        for neighbor in get_neighbors(current, satellite_positions):
            if tuple(neighbor) in visited_sats:
                continue
            tentative_g_cost = g_cost[tuple(current)] + edge_cost(current, neighbor)

            if tentative_g_cost < g_cost[tuple(neighbor)]:
                g_cost[tuple(neighbor)] = tentative_g_cost
                f_cost = tentative_g_cost + heuristic_cost_estimate(neighbor, target)
                heapq.heappush(open_set, (f_cost, neighbor))
                came_from[tuple(neighbor)] = tuple(current)

    return float('inf'), list(visited_sats), []  # no valid path

In [ ]:
import numpy as np


Re = 6371
altitude = 550  #

berlin_lat, berlin_lon = 52.52, 13.405
cape_town_lat, cape_town_lon = -33.9249, 18.4241




def line_of_sight(point1, point2, R_E=6371.8):

    point1 = np.array(point1)
    point2 = np.array(point2)

    point1_to_point2 = point2 - point1

    distance = np.linalg.norm(point1_to_point2)

    point1_distance = np.linalg.norm(point1)
    point2_distance = np.linalg.norm(point2)

    angle = np.arccos(np.dot(point1_to_point2, point1) / (distance * point1_distance))

    if angle < np.pi / 2 and distance > np.sqrt(point1_distance**2 - R_E**2) + np.sqrt(point2_distance**2 - R_E**2):
        return True  # los
    else:
        return False  # Earth blocks the los


berlin_pos = geodetic_to_eci(berlin_lat, berlin_lon, 0)
cape_town_pos = geodetic_to_eci(cape_town_lat, cape_town_lon, 0)
"""

num_satellites = 100
satellites_in_los_berlin = []
satellites_in_los_cape_town = []
satellites_eci = dict()
print(cape_town_pos)

for i in range(num_satellites):
    sat_lat = np.random.uniform(-90, 90)
    sat_lon = np.random.uniform(-180, 180)
    satellite_pos = geodetic_to_eci(sat_lat, sat_lon, altitude)
    satellites_eci[i]=satellite_pos
    if line_of_sight(satellite_pos, berlin_pos):
        satellites_in_los_berlin.append((sat_lat, sat_lon, altitude))

    if line_of_sight(satellite_pos, cape_town_pos):
        satellites_in_los_cape_town.append((sat_lat, sat_lon, altitude))
"""


num_satellites = 50
satellites_in_los_berlin = {}
satellites_in_los_cape_town = {}
satellites_eci = {}
#adding ground pomts to the dict eci q
satellites_eci[0]=berlin_pos
satellites_eci[-1]=cape_town_pos

for i in range(1,num_satellites-1):
    sat_lat = np.random.uniform(-90, 90)
    sat_lon = np.random.uniform(-180, 180)
    satellite_pos = geodetic_to_eci(sat_lat, sat_lon, altitude)
    satellites_eci[i] = satellite_pos  # Add sat to dict

    if line_of_sight(satellite_pos, berlin_pos):
        satellites_in_los_berlin[i]=(sat_lat, sat_lon, altitude)

    if line_of_sight(satellite_pos, cape_town_pos):
        satellites_in_los_cape_town[i]=(sat_lat, sat_lon, altitude)


source = berlin_pos
target = cape_town_pos


distance, visited_sats, path = a_star_routing(source, target, satellites_eci)


print(f"Distance: {distance} km")
print(f"Visited Satellites: {visited_sats}")
print(f"Path indicies: {path}")
for idx in path:

  print(f"Path (ECI coordinates): {satellites_eci[idx]}")

In [ ]:
import numpy as np

def delta_walker_constellation(P, S, altitude, inclination, R_E=6371.8):

    satellites = {}
    index = 0

    inclination_rad = np.radians(inclination)

    semi_major_axis = R_E + altitude
    delta_omega = 360 / P  #
    delta_nu = 360 / S     #

    for plane in range(P):
        omega = plane * delta_omega
        for sat in range(S):
            nu = sat * delta_nu
            x = semi_major_axis * (np.cos(np.radians(omega)) * np.cos(np.radians(nu)) -
                                   np.sin(np.radians(omega)) * np.sin(np.radians(nu)) * np.cos(inclination_rad))
            y = semi_major_axis * (np.sin(np.radians(omega)) * np.cos(np.radians(nu)) +
                                   np.cos(np.radians(omega)) * np.sin(np.radians(nu)) * np.cos(inclination_rad))
            z = semi_major_axis * (np.sin(np.radians(nu)) * np.sin(inclination_rad))
            satellites[index] = np.array([x, y, z])
            index += 1

    return satellites